In [1]:
#Database me missing value ko SQL ke through identify karna

In [2]:
#01:har column me NULL count nikalna(ek saath sab columns ke liye)

In [3]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../data/db/ecommerce.db')

q1 = pd.read_sql("""
    SELECT
        SUM(CASE WHEN Invoice IS NULL THEN 1 ELSE 0 END) as null_invoice,
        SUM(CASE WHEN StockCode IS NULL THEN 1 ELSE 0 END) as null_stockcode,
        SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) as null_description,
        SUM(CASE WHEN Quantity IS NULL THEN 1 ELSE 0 END) as null_quantity,
        SUM(CASE WHEN InvoiceDate IS NULL THEN 1 ELSE 0 END) as null_invoicedate,
        SUM(CASE WHEN Price IS NULL THEN 1 ELSE 0 END) as null_price,
        SUM(CASE WHEN "Customer ID" IS NULL THEN 1 ELSE 0 END) as null_customerid,
        SUM(CASE WHEN Country IS NULL THEN 1 ELSE 0 END) as null_country,
        COUNT(*) as total_rows
    FROM transactions
""", conn)
print(q1)

   null_invoice  null_stockcode  null_description  null_quantity  \
0             0               0              4382              0   

   null_invoicedate  null_price  null_customerid  null_country  total_rows  
0                 0           0           243007             0     1067371  


In [4]:
#ye confirm karega jo humne pehle python me dekha tha (Description:4382nulls, Customer ID:243007 nulls) - ab SQL se bhi wahi validate ho jaiga.

In [5]:
#02: Missing Customer ID ka percentage nikalna:

In [6]:
q2 = pd.read_sql("""
    SELECT
        SUM(CASE WHEN "Customer ID" IS NULL THEN 1 ELSE 0 END) as missing_count,
        COUNT(*) as total,
        ROUND(SUM(CASE WHEN "Customer ID" IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as missing_pct
    FROM transactions
""",conn)
print(q2)

   missing_count    total  missing_pct
0         243007  1067371        22.77


In [7]:
#03:Missing Customer ID kis country/pattern me jyada hai, dekhna:

In [8]:
q3 = pd.read_sql("""
    SELECT Country,
           COUNT(*) as total_rows,
           SUM(CASE WHEN "Customer ID" IS NULL THEN 1 ELSE 0 END) as missing_customer_id,
           ROUND(SUM(CASE WHEN "Customer ID" IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as missing_pct
    FROM transactions
    GROUP BY Country
    HAVING missing_customer_id > 0
    ORDER BY missing_pct DESC
""", conn)
print(q3)

                 Country  total_rows  missing_customer_id  missing_pct
0              Hong Kong         364                  364       100.00
1                Bermuda          34                   34       100.00
2                Bahrain         126                   67        53.17
3            Unspecified         756                  232        30.69
4                    RSA         169                   46        27.22
5         United Kingdom      981330               240029        24.46
6   United Arab Emirates         500                  114        22.80
7                Lebanon          58                   13        22.41
8                 Israel         371                   47        12.67
9                   EIRE       17866                 1671         9.35
10               Nigeria          32                    2         6.25
11              Portugal        2620                  116         4.43
12           Switzerland        3189                  125         3.92
13    

In [9]:
#04:Missing description wale rows dekhna(kya inka StockCode kuch pattern follow karta hai):

In [10]:
q4 = pd.read_sql("""
    SELECT StockCode, Quantity, Price, Country
    FROM transactions
    WHERE Description IS NULL
    LIMIT 20
""", conn)
print(q4)

   StockCode  Quantity  Price         Country
0      21646       -50    0.0  United Kingdom
1      20683       -44    0.0  United Kingdom
2      21350       230    0.0  United Kingdom
3      84292        17    0.0  United Kingdom
4      18010      -770    0.0  United Kingdom
5     85049G      -240    0.0  United Kingdom
6     35751C        12    0.0  United Kingdom
7     79323G       954    0.0  United Kingdom
8      21098      -200    0.0  United Kingdom
9      21166        48    0.0  United Kingdom
10     21982       467    0.0  United Kingdom
11     21982     -1012    0.0  United Kingdom
12     20620       -25    0.0  United Kingdom
13     85064       -89    0.0  United Kingdom
14    84508B       184    0.0  United Kingdom
15     21558      -169    0.0  United Kingdom
16     84347        80    0.0  United Kingdom
17     21493      -106    0.0  United Kingdom
18     21489       -23    0.0  United Kingdom
19     21490       -44    0.0  United Kingdom


In [11]:
#05:Empty string vs NULL ka fark check karna(kabhi kabhi missing data NULL nahi hota, blank string'' hota hai - yeh ek common data-quality issue hai):

In [12]:
q5 = pd.read_sql("""
    SELECT 
        SUM(CASE WHEN TRIM(Description) = '' THEN 1 ELSE 0 END) as empty_string_desc,
        SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) as null_desc
    FROM transactions
""", conn)
print(q5)

   empty_string_desc  null_desc
0                  0       4382


In [13]:
#

In [14]:
conn.close()